# New OCR parsing

Having extracted the receiptTexts that were matched with an Expense, I can improve and simplify the original `notebook-api-seed.ipynb` script. 

Objectives: 
- get high accuracy expense matching using fuzzy on strings from processed receipts
- if new vendors have low match rate: get as many matches as possible by parsing product descriptions against new receipts

Specs
- use processed receipts strings (cleaned up list) to parse expnses
- new list has product id that should be retained
- adjust API call to create expense
- might want to do 2 passes on matches, or even append product descriptions to the list of matches (assuming they match less); product id will need to be match

In [179]:
%pip install -q \
    fuzzywuzzy \
    easyocr \
    pandas \
    requests


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [180]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import re
import easyocr
import pandas as pd
import shutil
# import numpy as np
# import os
import json

In [181]:
import requests

def writeToEndpoint(endpoint, json, method='POST'):
    # url = f'http://localhost:3000/api/{endpoint}'
    url = f'http://localhost:3001/api/{endpoint}'
    response = requests.request(method, url, json=json)
    # print(response.text)
    if response.status_code in [200, 201]:
        print(f'Successful {method} on {endpoint}. Response:\n{response.text}')
    else:
        print(f'\nError {method} on {endpoint}. \nRequest:{json}\nResponse:\n{response.text}')
    return response

In [182]:
# import from known products to panda dataframe
products_df = pd.read_csv('data/receipt_texts_clean.csv')
choices_dict = products_df[['receipt_text']].to_dict()['receipt_text']

In [183]:
reader = easyocr.Reader(['en']) # this needs to run only once to load the model into memory

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [184]:
def ocrImage(image_id, image_path):
    receipt_strings = reader.readtext(image_path, width_ths=0.7)

    # guard clause
    if len(receipt_strings) < 1:
        return # no text found

    # transformation
    receipt_strings_df = pd.DataFrame(receipt_strings, columns=['boundingBox', 'text', 'confidence'])
    receipt_strings_df.drop(columns=['confidence'], inplace=True)
    receipt_strings_df['imageFileId'] = image_id

    # push to API
    responses = []
    for index, row in receipt_strings_df.iterrows():
        # print(row['text'])
        # print(row['boundingBox'])
        # print(row['imageFileId'])
        boundingBox_str = [[int(value) for value in sublist] for sublist in row['boundingBox']]
        response = writeToEndpoint('receiptText', {
            'text': str(row['text']), 
            'boundingBox': str(boundingBox_str), 
            'imageFileId': int(row['imageFileId'])
        })
        responses.append(response.json())
    return responses

import re
from dateutil.parser import parse
from datetime import datetime
def identify_receipt_date(receiptTexts):
    date_patterns = [
        # DD/MM/YYYY or DD/MM/YY
        r'\b(0?[1-9]|[12][0-9]|3[01])[-/\.](0?[1-9]|1[0-2])[-/\.](\d{4}|\d{2})\b',
        # MM/DD/YYYY or MM/DD/YY
        r'\b(0?[1-9]|1[0-2])[-/\.](0?[1-9]|[12][0-9]|3[01])[-/\.](\d{4}|\d{2})\b',
        # YYYY-MM-DD or YY-MM-DD
        r'\b(\d{4}|\d{2})[-/\.](0?[1-9]|1[0-2])[-/\.](0?[1-9]|[12][0-9]|3[01])\b',
        # Month name formats
        r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[-/\.\s]+(0?[1-9]|[12][0-9]|3[01])(st|nd|rd|th)?[-/\.\s,]*\s*(\d{4}|\d{2})\b',
        r'\b(0?[1-9]|[12][0-9]|3[01])(st|nd|rd|th)?[-/\.\s]+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[-/\.\s,]*\s*(\d{4}|\d{2})\b',
    ]

    potential_dates = []
    def extract_dates(text):
        for pattern in date_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                try:
                    # Use the entire matched string
                    date_str = match.group(0)
                    date_obj = parse(date_str, fuzzy=True)
                    
                    potential_dates.append({
                        'date': date_obj,
                        'original_text': date_str,
                        'pattern': pattern
                    })
                except (ValueError, TypeError) as e:
                    continue
        return potential_dates
        
    for receiptText in receiptTexts:
        text = receiptText['text']
        extract_dates(text)

    # Sort by confidence and return best match
    if potential_dates:
        current_year = datetime.now().year
        year_counts = {}
        for date_info in potential_dates:
            year = date_info['date'].year
            if year < (current_year - 10) or year > current_year:
                continue
            year_counts[year] = year_counts.get(year, 0) + 1
        # Find the year with the most occurrences
        if year_counts:
            # Return the most recent date from the most likely year
            most_likely_year = 2024 #max(year_counts.items(), key=lambda x: x[1])[0] # TODO: remove quick and dirty fix
            # Filter dates to only those from the most likely year
            filtered_dates = [d for d in potential_dates if d['date'].year == most_likely_year]
            # Sort by precision (more components = more precise) and then by recency
            filtered_dates.sort(key=lambda x: (
                -sum(1 for attr in ['year', 'month', 'day', 'hour', 'minute', 'second'] 
                     if getattr(x['date'], attr) is not None),
                x['date']
            ))
            if len(filtered_dates) > 0:
                return filtered_dates[0]['date']
            # return max(filtered_dates, key=lambda x: x['date'])['date']
    return None

def parseReceiptTextsForEligibleExpenses(receipt_texts, receipt_id):
    # receipt = fetch_receipt(receipt_id)
    # receipt_texts = receipt['data']['receipt']['receiptTexts']
    threshold = 95
    
    def clean_text(text):
        text = re.sub(r'^\d{1,10}', '', text)
        if len(text) <= 2:
            return ''
        if not any(char.isalnum() for char in text):
            return ''
        return text
    
    def getPriceFromReceiptText(receipt_text_id, receipt_texts=receipt_texts):
        # 1. get bounding box of receipt text id
        receipt_text = next(filter(lambda x: x['id'] == receipt_text_id, receipt_texts))
        bounding_box = json.loads(receipt_text['boundingBox']) # "[[165, 910], [657, 910], [657, 959], [165, 959]]"
        # geometries
        # 1 get height of current text (x3 for search bound)
        # x_values = [coord[0] for coord in bounding_box]
        y_values = [coord[1] for coord in bounding_box]
        # width  = max(x_values) - min(x_values)
        height = max(y_values) - min(y_values)
        buffer = 0.25*2 # 0.25 for each side
        search_box_height = 3 * height
        search_box_y = min(y_values) - buffer * height
        # for simplicity search whole width, so no need for x values
        # 2 get all receiptTexts within search bound
        def filter_candidates(y_min, y_max, receipt_texts=receipt_texts):
            candidate_receiptTexts = []
            for receipt_text_i in receipt_texts:
                bounding_box_i = json.loads(receipt_text_i['boundingBox'])
                y_values_i = [coord[1] for coord in bounding_box_i]
                if all(y_min <= y <= y_max for y in y_values_i):
                    candidate_receiptTexts.append(receipt_text_i)
            return candidate_receiptTexts
        candidate_receiptTexts = filter_candidates(search_box_y, search_box_y + search_box_height)
        def find_price_from_candidate_receiptTexts(candidate_receiptTexts):
            for candidate_receiptText in candidate_receiptTexts:
                text = candidate_receiptText['text']
                price = re.findall(r'\d+\.\d{2}', text)
                if len(price) > 0:
                    return float(price[0])
            return None
        price = find_price_from_candidate_receiptTexts(candidate_receiptTexts)
        return price

    for receipt_text in receipt_texts:
        receipt_text_id = receipt_text['id']
        receipt_text = receipt_text['text']
        clean_receipt_text = clean_text(receipt_text)
        if clean_text(clean_receipt_text) == '':
            continue
        best_match, best_score, best_match_index = process.extractOne(clean_receipt_text, choices_dict, scorer=fuzz.token_set_ratio)

        if best_score >= threshold:
            # avoid strict substring matches
            if len(clean_receipt_text)/len(best_match) < 0.4:
                continue
            # found a decent match
            # get corresponding referenceItem 
            product_id = int(products_df.iloc[best_match_index]['product_id'])
            # get price
            item_price_each = getPriceFromReceiptText(receipt_text_id) or 0
            # create eligible expense
            # print(f'Creating expense for {receipt_text} with reference item id {reference_item_id}')
            writeToEndpoint('expense',{
                    'receiptTextId': receipt_text_id,
                    'productId': product_id,
                    'receiptId': receipt_id,
                    'priceEach': item_price_each, # TODO: find price
                    'quantity': 1, # TODO: find quantity
                }
            )

def upload_image(image_path, receipt_id):
    # fake upload, i.e. copy to frontend folder
    local_destination_folder = "../frontend/public/uploads/"
    remote_destination_folder = "/uploads/"
    shutil.copy(image_path, local_destination_folder)
    image_url = remote_destination_folder + image_path.split("/")[-1]
    response = writeToEndpoint('imageFile', {
        "url": image_url, 
        "receiptId": receipt_id,
        })
    return response
# test
# upload_image('./receipts/IMG_4553.jpg')

# TODO: batch multiple images in array
def main_single_receipt(image_path):
    receipt_id = writeToEndpoint('receipt', {
        'receiptDate': '2024-01-01T00:00:00.000Z'
    }).json()['id']
    image_id = upload_image(image_path, receipt_id).json()['id']
    receiptTexts = ocrImage(image_id, image_path) # for notebooks only, using local file
    # print(f'Parsed receipt {receipt_id}: {receiptTexts}', end='\r')
    if receiptTexts is None:
        return None
    receipt_date = identify_receipt_date(receiptTexts)
    if receipt_date:
        print(f'Receipt date: {receipt_date}')
        writeToEndpoint('receipt', {
            'receiptId': int(receipt_id),
            'receiptDate': receipt_date.strftime('%Y-%m-%d %H:%M:%S')
        }, method='PUT')
    # else:
    #     print('No receipt date found')
    parseReceiptTextsForEligibleExpenses(receiptTexts, receipt_id)

In [185]:
# main_single_receipt('./new_receipt_test.jpg')

In [186]:
# scorers = [
#     ('ratio', fuzz.ratio),
#     ('partial_ratio', fuzz.partial_ratio),
#     ('token_sort_ratio', fuzz.token_sort_ratio),
#     ('token_set_ratio', fuzz.token_set_ratio),
#     ('partial_token_sort_ratio', fuzz.partial_token_sort_ratio),
#     ('partial_token_set_ratio', fuzz.partial_token_set_ratio),
#     ('WRatio', fuzz.WRatio),
# ]

# for scorer_name, scorer_func in scorers:
#     matches = process.extract(
#         "NATURE $ PATH PU", 
#         choices_dict, 
#         scorer=scorer_func,
#         limit=None
#     )
#     print(f"\n{scorer_name}:")
#     for match, score, match_id in matches:
#         print(f"  {score} - {choices_dict[match_id]}")

In [187]:
import os

def list_files(directory, extension):
    return list(f for f in os.listdir(directory) if f.endswith('.' + extension))

files_dir = './.temp/2024/raw'
all_files = list_files(files_dir, 'jpg')

for file in all_files:
    # main_single_receipt(f'./receipts/{file}')
    main_single_receipt(f'{files_dir}/{file}')

Successful POST on receipt. Response:
{"id":106,"createdAt":"2025-04-18T22:54:45.725Z","updatedAt":"2025-04-18T22:54:45.725Z","receiptDate":"2024-01-01T00:00:00.000Z"}
Successful POST on imageFile. Response:
{"id":106,"url":"/uploads/001_Receipt-2024-1 (1).jpg","receiptId":106,"createdAt":"2025-04-18T22:54:45.747Z","updatedAt":"2025-04-18T22:54:45.747Z"}
Successful POST on receiptText. Response:
{"id":12246,"text":"Mill Street Crepe Co _","boundingBox":"[[240, 390], [656, 390], [656, 444], [240, 444]]","imageFileId":106,"createdAt":"2025-04-18T22:55:00.112Z","updatedAt":"2025-04-18T22:55:00.112Z"}
Successful POST on receiptText. Response:
{"id":12247,"text":"14 MilI Street Unit","boundingBox":"[[244, 437], [631, 437], [631, 494], [244, 494]]","imageFileId":106,"createdAt":"2025-04-18T22:55:00.123Z","updatedAt":"2025-04-18T22:55:00.123Z"}
Successful POST on receiptText. Response:
{"id":12248,"text":"1","boundingBox":"[[649, 445], [663, 445], [663, 477], [649, 477]]","imageFileId":106,"c